# Method 02 — Primary Same-Endpoint Comparison

Compare Australia's change with supplied-country comparators that report the same exact endpoints. Native changes are retained for interpretation and lower-is-better outcomes are direction-oriented for ranking.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

SUBMISSION_ROOT = Path.cwd() / 'submission' if (Path.cwd() / 'submission' / 'OECD Data.csv').exists() else (Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd())
sys.path.insert(0, str(SUBMISSION_ROOT.parent))
from submission.code.oecd_audit import INDICATOR_SPECS, load_clean

OUTCOME_SPECS = [
    ('1_1', 2010, 2024, '2010–2024'),
    ('2_1', 2010, 2024, '2010–2024'),
    ('7_1_DEP', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
    ('11_2', 2010, 2024, '2008–10 to 2023–25 pooled windows'),
]
TABLE_DIR = SUBMISSION_ROOT / 'report' / 'tables'

def endpoint_changes(data, code, start, end):
    references = sorted(set(data.country_code) - {'AUS'})
    subset = data.loc[
        data.indicator_code.eq(code)
        & data.country_code.isin(['AUS', *references])
        & data.year.isin([start, end])
    ].copy()
    subset = subset.sort_values('year').drop_duplicates(
        ['country_code', 'independent_period'], keep='last'
    )
    wide = (subset.pivot(index='country_code', columns='year', values='value')
            .reindex(columns=[start, end]).dropna().reset_index()
            .rename(columns={start: 'start_value', end: 'end_value'}))
    wide['absolute_change'] = wide.end_value - wide.start_value
    sign = 1 if INDICATOR_SPECS[code].direction == 'higher' else -1
    wide['oriented_change'] = wide.absolute_change * sign
    return wide

def build_primary_results(data):
    rows = []
    for code, start, end, period_label in OUTCOME_SPECS:
        changes = endpoint_changes(data, code, start, end)
        if 'AUS' not in changes.country_code.values:
            raise ValueError(f'Australia lacks a required endpoint for {code}.')
        australia = changes.loc[changes.country_code.eq('AUS')].iloc[0]
        comparators = changes.loc[changes.country_code.ne('AUS')]
        raw = data.loc[data.country_code.eq('AUS') & data.indicator_code.eq(code)
                       & data.year.isin([start, end])].sort_values('year')
        if raw.year.tolist() != [start, end]:
            raise ValueError(f'Australia lacks both displayed endpoints for {code}.')
        rank = changes.oriented_change.rank(ascending=False, method='average').loc[australia.name]
        rows.append({
            'indicator_code': code, 'indicator': raw.iloc[0].indicator, 'unit': raw.iloc[0].unit,
            'comparison_period': period_label, 'start_year_displayed': start, 'end_year_displayed': end,
            'start_independent_period': raw.iloc[0].independent_period,
            'end_independent_period': raw.iloc[1].independent_period,
            'better_direction': INDICATOR_SPECS[code].direction,
            'australia_start_value': australia.start_value, 'australia_end_value': australia.end_value,
            'australia_native_change': australia.absolute_change,
            'native_change_definition': 'end value minus start value; natural unit and sign retained',
            'comparator_median_native_change': comparators.absolute_change.median(),
            'australia_minus_comparator_median_oriented': australia.oriented_change - comparators.oriented_change.median(),
            'oriented_gap_definition': 'positive means more favourable Australian change',
            'australia_favourable_percentile': 100 * (len(changes) - rank) / (len(changes) - 1),
            'eligible_comparator_country_count': len(comparators),
        })
    return pd.DataFrame(rows)

In [ ]:
data = load_clean()
results = build_primary_results(data)
assert results.indicator_code.tolist() == [spec[0] for spec in OUTCOME_SPECS]
assert results.eligible_comparator_country_count.tolist() == [31, 43, 46, 46]
assert results.australia_favourable_percentile.between(0, 100).all()
assert results.better_direction.tolist() == ['higher', 'higher', 'lower', 'lower']
expected = {'1_1': (44625.0, 50629.0), '2_1': (75.444, 80.262),
            '7_1_DEP': (4.926543, 10.043193), '11_2': (12.244020, 14.853043)}
for row in results.itertuples(index=False):
    start, end = expected[row.indicator_code]
    assert abs(row.australia_start_value - start) < 1e-5
    assert abs(row.australia_end_value - end) < 1e-5
TABLE_DIR.mkdir(parents=True, exist_ok=True)
output_path = TABLE_DIR / 'material_social_primary_results.csv'
results.to_csv(output_path, index=False)
pd.testing.assert_frame_equal(results, pd.read_csv(output_path), check_dtype=False,
                              check_exact=False, rtol=1e-12, atol=1e-12)
display(results)
print(f'Wrote and validated {output_path.relative_to(SUBMISSION_ROOT)}')